In [1]:
%load_ext autoreload
%autoreload 2
 
import sys, pathlib, json, pprint, pandas as pd 
from pathlib import Path
from pydantic import Field,BaseModel 
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from runtime.v4.semantics.load_semantics import load_semantics, load_idiom_rules
from runtime.v4.semantics.semantic_models import * 
from runtime.v4.analyst_agent.catalog import Catalog
from runtime.v4.analyst_agent.smart_data import SmartData
from runtime.v4.analyst_agent.smart_data_tools  import SmartDataTools
from runtime.v4.analyst_agent.analyst_models  import *
from runtime.v4.analyst_agent.analyst_prompts import system_prompt_template3 as agent_system_prompt


import  os  
from datetime import datetime 
from langchain_core.tools import StructuredTool, Tool


import duckdb
from typing import Any, Dict, List, Iterable, Union, Optional 
import yaml
import pprint
import pandas as pd, numpy as np
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig, Runnable, RunnableLambda
 
from langchain.tools import tool, ToolRuntime
from langgraph.runtime import get_runtime 
from langchain.agents import create_agent
from langchain_core.runnables import RunnableLambda
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_core.runnables import Runnable
from langchain.tools import tool, ToolRuntime
from langgraph.runtime import get_runtime 
from langchain.agents import create_agent
 

In [2]:
inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year
pinj = pd.read_csv("../datasets/IX5I_4P/producers.csv")
pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
pinj['DAY']   = pinj['DATE'].dt.day
pinj['MONTH'] = pinj['DATE'].dt.month
pinj['YEAR']  = pinj['DATE'].dt.year
locs= pd.read_csv("../datasets/IX5I_4P/locations.csv")
 
from get_llm_model import azure_llm_if
from support import * 

imported


In [3]:

semantics_json = Path('../runtime/v4/semantics/semantic_models.json')
semantic_catalog = load_semantics(semantics_json)
known_table_models = { t.name: t for t in semantic_catalog.tables } 

df_dict = {'injectors': inj, 'producers':pinj , 'locations': locs }
data = SmartData()
data.initialize_from_named_dataframes( df_dict, known_table_models)

smart_data_tools = SmartDataTools( data=data)
tools = smart_data_tools.get_tools(include_planning_tools=False)
print(smart_data_tools.catalog_snapshot())


idiom_rules_path = Path('../runtime/v4/semantics/idioms.json')
idiom_rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = idiom_rules_path )
idiom_rules

------------------------------------------------------------

Table: injectors
  description: Injector water injection time series.
  kind: base
  row_count: 490
  Columns:
    - name: DATE | data_type: timestamp | description: Injection date. | derived_column: False
    - name: NAME | data_type: string | description: Injector well identifier. | derived_column: False
    - name: WATER_INJECTION_VOLUME | data_type: float | description: Injected water volume. | derived_column: False
    - name: SUBZONE | data_type: string | description: Vertical subzone. | derived_column: False
    - name: SECTOR | data_type: integer | description: Geographic sector. | derived_column: False
    - name: YEAR | data_type: integer | description: Year from DATE. | derived_column: False
    - name: MONTH | data_type: integer | description: Month from DATE. | derived_column: False
    - name: DAY | data_type: integer | description: Day from DATE. | derived_column: False

Table: producers
  description: Produce

{'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
 'date truncation': "Use DATE_TRUNC('month', column).",
 'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
 'string concatenation': 'Use the || operator or CONCAT().',
 'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
 'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
 'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ..

In [10]:
constraints = "".join([f"- {i}\n" for i in semantic_catalog.semantic_constraints])
print(constraints)

idiom_context = "".join([f"- {i}: {v}\n" for i,v in idiom_rules.items()])
print( idiom_context )



- WELL_TYPE domain is restricted to exactly two values: 'Injector' and 'Producer'.
- Each well belongs to exactly one WELL_TYPE category (mutually exclusive).
- Well identifiers are stored in NAME and used consistently as join keys across tables.
- injectors.NAME joins to locations.NAME only for rows where locations.WELL_TYPE = 'Injector'.
- producers.NAME joins to locations.NAME only for rows where locations.WELL_TYPE = 'Producer'.
- All volume measures are non-negative (WATER_INJECTION_VOLUME, LIQUID_VOLUME, WATER_VOLUME, OIL_VOLUME, GAS_VOLUME).
- Production balance constraint: LIQUID_VOLUME is approximately WATER_VOLUME + OIL_VOLUME + GAS_VOLUME (allowing small numerical tolerance).
- If YEAR, MONTH, DAY are present, they should match the corresponding DATE components.

- date subtraction: Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().
- date truncation: Use DATE_TRUNC('month', column).
- reserved keywords: Always wrap the column name "DATE" in double quo

# Direct ReAct agent. Simple

In [11]:

single_agent_data = SmartData()
single_agent_data.initialize_from_named_dataframes( df_dict, known_table_models )
single_agnet_tools = SmartDataTools( single_agent_data ).get_tools()

#from runtime.v4.analyst_agent.analyst_prompts import system_prompt_template3 as system_prompt_template3
from runtime.v4.analyst_agent.analyst_prompts import system_prompt_template1 as system_prompt_template1
#prompt = system_prompt_template3.format( idiom = 'duckdb', idiom_examples = idiom_context, constraints = constraints)
prompt = system_prompt_template1.format( idiom = 'duckdb', idiom_examples = idiom_context, constraints = constraints)
print( prompt )



You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database

Workflow:
You must:
1. Always call catalog_snapshot first.

2. Analyze the question and the information in the catalog and produce a concise PLAN
The PLAN must be concise and must include:
- required source tables
- whether existing derived tables can be reused
- target table names to materialize
- high-level transformation logic, without SQL


3. You MUST record the PLAN in plain text. Only after the PLAN message is sent may you call sql_* tools.
4. Use sql_materialize to create intermediate tables.
5. When multiple output tables are to be produced, proceed sequentially one at a time  
6. Your job finishes once all the target tables are confirmed present (either via initial audit or your materializations).  

Important:
- The name of generated tables and columns should reflect the table contents  
- Use lowercase snake_case for table names and column name

In [12]:

llm = azure_llm_if()

single_agent_data.clear_derived()
agent = create_agent(
        model=llm,
        system_prompt=prompt,
        tools=single_agnet_tools,
        response_format=AgentTableResponse,
        #checkpointer= MemorySaver() 
    )

messages = {"messages": [{"role": "user", "content": query11}]}
response = run_agent_stream_values(agent, messages )

#name = response['structured_response'].tables[0].table_name
#data.get_table_as_df( name )


--- HumanMessage ---
name: None
Create two separate tables: one ranking injector wells by total water injection volume, and another ranking producer wells by total oil production volume.

--- AIMessage ---
name: None
TOOL CALL: catalog_snapshot
ARGS: {}

--- ToolMessage ---
name: catalog_snapshot
------------------------------------------------------------

Table: injectors
  description: Injector water injection time series.
  kind: base
  row_count: 490
  Columns:
    - name: DATE | data_type: timestamp | description: Injection date. | derived_column: False
    - name: NAME | data_type: string | description: Injector well identifier. | derived_column: False
    - name: WATER_INJECTION_VOLUME | data_type: float | description: Injected water volume. | derived_column: False
    - name: SUBZONE | data_type: string | description: Vertical subzone. | derived_column: False
    - name: SECTOR | data_type: integer | description: Geographic sector. | derived_column: False
    - name: YEAR | d

In [13]:
table_name = response['structured_response'].tables[0].table_name
print(table_name )
single_agent_data.get_table_as_df( table_name )



ranked_injector_wells_by_water_injection


,NAME,total_water_injection_volume,rank
0,I1,157996.408,1
1,I2,125508.395,2
2,I5,96638.092,3
3,I3,82427.218,4
4,I4,75614.869,5


# As a graph, tool or node. 

In [ ]:
from langgraph.graph import StateGraph, END

def executor_prompt_builder_from_structured_plan(
    idiom: str,
    idiom_rules: dict,
    structured_plan,
    user_query: str = "" 
) -> str:

    step_blocks = []
    for step in structured_plan.steps:

        source_tables = ", ".join(step.source_tables)
        reusable_tables = (
            ", ".join(step.reusable_tables)
            if step.reusable_tables
            else "None"
        )

        block = f"""
        Step {step.step_id}
        Target Table: {step.target_table}
        Source Tables: {source_tables}
        Reusable Tables: {reusable_tables}
        Logic: {step.logic}
        """.strip()

        step_blocks.append(block)
        
    formatted_plan = "\n\n".join(step_blocks)
    idiom_examples = "\n".join(
            [f"- {k}: {v}" for k, v in idiom_rules.items()]
        )
 
    # -----------------------------------------
    # Build final system prompt
    # -----------------------------------------
    prompt = system_prompt_sql_executor_template.format(
        idiom=idiom,
        idiom_examples=idiom_examples,
        plan=formatted_plan
    )

    if user_query:
        prompt = prompt + f"\n\n**USER QUERY**:\n{user_query}\n"
        
    return prompt

def planner_prompt_builder( tools:SmartDataTools )->str:
    #txt = data.catalog_snapshot()
    #txt = json.dumps( data.catalog_snapshot(), indent=3)
    txt = tools.catalog_snapshot()
    prompt = system_prompt_sql_planner_template.format(catalog=txt)

    return prompt 

class DataAnalystState(BaseModel):
    user_query: str
    refined_query: Optional[str] = None  
    plan: Optional[ExecutionPlan] = Field(
        default=None,
        description="Structured execution plan generated by planner",
    )

    execution_result: Optional[AgentTableResponse]  = Field(
        default=None,
        description="Executor output",
    )

    error: Optional[str] = Field(
        default=None,
        description="Execution or planning error",
    )

class DataAnalyst:

    def __init__(self, llm, tables_dict: None | Dict[str,pd.DataFrame] = None, 
                            known_table_models: None | Dict[str,TableCard] = None ):

        self._data  : SmartData #= SmartData()
        self._tools : List[StructuredTool] #= SmartDataTools( self._data ).get_tools()
        self._llm = llm 
        self._idiom = 'duckdb'
        self._graph = None 

    
        self._idiom_rules= {'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
        'date truncation': "Use DATE_TRUNC('month', column).",
        'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
        'string concatenation': 'Use the || operator or CONCAT().',
        'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
        'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
        'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)'
        }

        if tables_dict and known_table_models:
            self.initialize_from_known_tables( tables_dict, known_table_models)


    def get_result_as_dataframes(
        self,
        state: DataAnalystState | dict,
    ) -> Dict[str, pd.DataFrame]:
        """
        Retrieve all materialized result tables as DataFrames.

        Supports:
        - DataAnalystState
        - raw graph dict output
        """
        # -----------------------------------------------------
        # Normalize state
        # -----------------------------------------------------
        if isinstance(state, dict):
            state = DataAnalystState(**state)

        results = {}

        if state.execution_result is None:
            return results

        if not state.execution_result.tables:
            return results

        # -----------------------------------------------------
        # Load tables
        # -----------------------------------------------------
        for table_item in state.execution_result.tables:
            table_name = table_item.table_name

            try:
                results[table_name] = self._data.get_table_as_df(
                    table_name
                )

            except Exception as e:
                print(
                    f"Failed loading table '{table_name}': {str(e)}"
                )

        return results

    def initialize_from_known_tables( self, 
                                     tables_dict: Dict[str,pd.DataFrame], 
                                     known_table_models: Dict[str,TableCard] ):
   
        self._data = SmartData() 
        self._data.initialize_from_named_dataframes( tables_dict, known_table_models)
        self._smart_data_tools = SmartDataTools( self._data )#.get_tools()
        self._tools = self._smart_data_tools.get_tools() 
        
    def planner_node( self, state:DataAnalystState):
        
        # generate structured plan
        messages = [] 
        instruction = state.user_query
        planner_prompt = planner_prompt_builder(self._smart_data_tools)
        print(planner_prompt)
        llm = self._llm
        try: 
            messages = [{
                "role": "system",
                "content": planner_prompt
                },
                {
                "role": "user",
                "content": instruction
                }]

            # Structured planner
            structured_llm = llm.with_structured_output(ExecutionPlan)
            plan = structured_llm.invoke(messages)
            #print( plan )


            return state.model_copy(
                update={
                    "plan": plan,
                    #"error": None,
                })


            #return {
            #    #"catalog": catalog,
            #    "plan": plan#.dict() if hasattr(plan, "dict") else plan,
            #}
        
        except Exception as e:
            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })

  
    def execution_node( self, state:DataAnalystState):
  
        llm = self._llm
        plan = state.plan
        idiom= self._idiom
        idiom_rules = self._idiom_rules

        q = plan.user_query if plan.refined_query is None else  plan.refined_query

        executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan,q )

        print("\n" + "=" * 80)
        print("EXECUTOR PROMPT")
        print("=" * 80)
        print(executor_prompt)
        print("=" * 80 + "\n")
        

        agent = create_agent(
                model=llm,
                system_prompt=executor_prompt,
                tools=self._tools,
                response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )
        
        try:
            response = run_agent_stream_values(agent, {} ) 
        

            #print('******************************')
            #print(response)
            #print('******************************')
            


            #return {
            ##"catalog": catalog,
            #"execution_result": response#.dict() if hasattr(plan, "dict") else plan,
            #} 
        
            return state.model_copy(
            update={
                "execution_result": response['structured_response'],
                "error": None,
            })

        
        except Exception as e:
            print("STREAM FAILED:", str(e))


            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })



            #return {
            #"catalog": catalog,
            #"error": str(e)#.dict() if hasattr(plan, "dict") else plan,
            #} 
            
        
    # Conditional after planner
    def should_continue_node(self, state: DataAnalystState):
        if state.error:
            return END

        if state.plan is None:
            return END

        if hasattr(state.plan, "steps") and not state.plan.steps:
            return END

        return "execution"

    # ------------------------------------------------------------------
    # Graph builder
    # ------------------------------------------------------------------
    def as_graph(self):
        builder = StateGraph(DataAnalystState)

        builder.add_node("planner", self.planner_node)
        builder.add_node("execution", self.execution_node)

        builder.set_entry_point("planner")

        builder.add_conditional_edges(
            "planner",
            self.should_continue_node,
            {
                "execution": "execution",
                END: END,
            },
        )

        builder.add_edge("execution", END)

        self._graph = builder.compile()

        return self._graph

    # ------------------------------------------------------------------
    # Main runner
    # ------------------------------------------------------------------
    def run(self, user_query: str):
        """
        Execute full planner -> executor workflow.

        Parameters
        ----------
        user_query : str
            Natural language analytical request.

        Returns
        -------
        DataAnalystState
            Final validated workflow state.
        """
        if self._graph is None:
            self.as_graph()

        initial_state = DataAnalystState(
            user_query=user_query,
            refined_query=None,
            plan=None,
            execution_result=None,
            error=None,
        )

        result = self._graph.invoke(initial_state)
        return self.normalize_state(result)

        # LangGraph may return dict depending on version
        #if isinstance(result, dict):
        #    return DataAnalystState(**result)
        #return result

# ------------------------------------------------------------------
# Normalize output
# ------------------------------------------------------------------
    def normalize_state(self, result) -> DataAnalystState:
        """
        Convert graph output into validated DataAnalystState.
        """
        if isinstance(result, DataAnalystState):
            return result

        if isinstance(result, dict):
            return DataAnalystState(**result)

        raise TypeError(
            f"Unsupported graph output type: {type(result)}"
        )


    # ------------------------------------------------------------------
    # Direct graph invoke wrapper
    # ------------------------------------------------------------------
    def invoke_graph(self, user_query: str) -> DataAnalystState:
        """
        Direct graph call but always returns structured state.
        """
        if self._graph is None:
            self.as_graph()

        result = self._graph.invoke(
            DataAnalystState(
                user_query=user_query,
                refined_query=None,
                plan=None,
                execution_result=None,
                error=None,
            )
        )

        return self.normalize_state(result)


    # ------------------------------------------------------------------
    # Export as LangChain tool
    # ------------------------------------------------------------------
    def as_tool(self) -> StructuredTool:
        """
        Expose DataAnalyst as a reusable tool for other agents.
        """

        def _run_analysis(user_query: str) -> dict:
            state = self.run(user_query)
            return state 
        
            return {
                "plan": (
                    state.plan.model_dump()
                    if state.plan else None
                ),
                "execution_result": (
                    state.execution_result.model_dump()
                    if state.execution_result else None
                ),
                "error": state.error,
            }

        return StructuredTool.from_function(
            func=_run_analysis,
            name="data_analyst",
            description=(
                "Executes structured analytical workflows over known tabular datasets. "
                "Useful for SQL-style table generation, ranking, aggregations, "
                "time-series analysis, and derived table creation."
            ),
        )


In [ ]:
analyst = DataAnalyst( llm, df_dict, known_table_models )


In [ ]:

tool = analyst.as_tool()
print(tool)

# =========================================================
# Direct tool invocation
# =========================================================
tool_result = tool.invoke(
    {
        "user_query": (
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    }
)
print("TOOL RESULT:")
type(tool_result)
tool_result.execution_result.tables

In [ ]:
dfs = analyst.get_result_as_dataframes(tool_result)
print(dfs.keys())
first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)



In [ ]:

graph = analyst.as_graph() 
display( graph )

# =========================================================
# Direct graph invoke
# =========================================================
result = graph.invoke(
    DataAnalystState(
        user_query=(
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    )
)

# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)





In [ ]:



# =========================================================
# Simple invoke test
# =========================================================
response = analyst.run(
    "Create two separate tables: "
    "one ranking injector wells by total water injection volume, "
    "and another ranking producer wells by total oil production volume."
)

print("FINAL RESPONSE:")
# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)

# End 


In [ ]:

llm = azure_llm_if()



planner_prompt = planner_prompt_builder(smart_data_tools)
print(planner_prompt)


In [ ]:
messages = [] 
instruction = query11


messages = [{
    "role": "system",
    "content": planner_prompt
    },
    {
    "role": "user",
    "content": instruction
    }]


plan = llm.invoke(messages).content

print( plan )

In [ ]:


idiom = 'duckdb'
idiom_rules= {'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
 'date truncation': "Use DATE_TRUNC('month', column).",
 'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
 'string concatenation': 'Use the || operator or CONCAT().',
 'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
 'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
 'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)'
}
executor_prompt = executor_prompt_builder(idiom, idiom_rules,plan)
agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        #response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 

In [ ]:
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))



# Add structure to the plan 


In [ ]:


# generate structured plan
messages = [] 
instruction = query11


messages = [{
    "role": "system",
    "content": planner_prompt
    },
    {
    "role": "user",
    "content": instruction
    }]

# Structured planner
structured_llm = llm.with_structured_output(ExecutionPlan)
plan = structured_llm.invoke(messages)
print( plan )






In [ ]:
def executor_prompt_builder_from_structured_plan(
    idiom: str,
    idiom_rules: dict,
    structured_plan,
    user_query: str = "" 
) -> str:

    step_blocks = []
    for step in structured_plan.steps:

        source_tables = ", ".join(step.source_tables)
        reusable_tables = (
            ", ".join(step.reusable_tables)
            if step.reusable_tables
            else "None"
        )

        block = f"""
        Step {step.step_id}
        Target Table: {step.target_table}
        Source Tables: {source_tables}
        Reusable Tables: {reusable_tables}
        Logic: {step.logic}
        """.strip()

        step_blocks.append(block)
        
    formatted_plan = "\n\n".join(step_blocks)
    idiom_examples = "\n".join(
            [f"- {k}: {v}" for k, v in idiom_rules.items()]
        )
 
    # -----------------------------------------
    # Build final system prompt
    # -----------------------------------------
    prompt = system_prompt_sql_executor_template.format(
        idiom=idiom,
        idiom_examples=idiom_examples,
        plan=formatted_plan
    )

    if user_query:
        prompt = prompt + f"\n\nThis is the user query:\n{user_query}"
        
    return prompt


executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan)


print(executor_prompt)



In [ ]:
executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan)


print(executor_prompt)

agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        #response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))





# Add structure to the output 

In [ ]:

llm = azure_llm_if()


In [ ]:
class PlanStep(BaseModel):
    step_id: int
    target_table: str
    source_tables: List[str]
    reusable_tables: List[str] = Field(default_factory=list)
    logic: str

class ExecutionPlan(BaseModel):
    #raw_query: str 
    user_query: str
    tables_needed: List[str] = Field( default=[],description="List of all the tables in the catalog that will be needed to answer the question")
    steps: List[PlanStep]



class TableItemAgentResponse(BaseModel):
    table_name: str = Field(description="Name of a materialized output table")
    description: str = Field(description="Brief summary of the table contents")
        
   
      
class AgentTableResponse(BaseModel):
    # Literal ensures the LLM chooses only these specific strings
    agent: Literal["analyst"] = Field(
        default="analyst", 
        description="The role of the agent. Always 'analyst'."
    )
    tables: List[str] = Field(default=[], description="Comma-separated list of table names")

    #text : Optional[str]  = Field(default=None, description="textual response")
    #tables: List[TableItemAgentResponse] = Field(default_factory=list, description="List of materialized output tables")
    
    

In [ ]:
agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))




# Prompt-chaining implementation 

In [ ]:
from typing import TypedDict
from typing import TypedDict, Optional, Dict, Any, Callable
from pydantic import BaseModel, Field
from typing import List
import sys, pathlib, json, pprint, pandas as pd 
from pathlib import Path
from pydantic import Field,BaseModel 
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from runtime.v4.semantics.load_semantics import load_semantics
from runtime.v4.semantics.semantic_models import * 
from runtime.v4.analyst_agent.catalog import Catalog
from runtime.v4.analyst_agent.smart_data import SmartData
from runtime.v4.analyst_agent.smart_data_tools  import SmartDataTools


from langgraph.graph import StateGraph, END



class PlanStep(BaseModel):
    step_id: int
    target_table: str
    source_tables: List[str]
    reusable_tables: List[str] = Field(default_factory=list)
    logic: str

class ExecutionPlan(BaseModel):
    #raw_query: str 
    user_query: str
    refined_query: Optional[str] = None 
    tables_needed: List[str] = Field( default=[],description="List of all the tables in the catalog that will be needed to answer the question")
    steps: List[PlanStep]

class TableItemAgentResponse(BaseModel):
    table_name: str = Field(description="Name of a materialized output table")
    description: str = Field(description="Brief summary of the table contents")
        

class AgentTableResponse(BaseModel):
    # Literal ensures the LLM chooses only these specific strings
    agent: Literal["analyst"] = Field(
        default="analyst", 
        description="The role of the agent. Always 'analyst'."
    )
    user_query: str = Field( description='sanitized user query')
    tables: List[TableItemAgentResponse] = Field(default=[], description="Comma-separated list of table names")

    #text : Optional[str]  = Field(default=None, description="textual response")
    #tables: List[TableItemAgentResponse] = Field(default_factory=list, description="List of materialized output tables")
    
    

In [ ]:



def executor_prompt_builder_from_structured_plan(
    idiom: str,
    idiom_rules: dict,
    structured_plan,
    user_query: str = "" 
) -> str:

    step_blocks = []
    for step in structured_plan.steps:

        source_tables = ", ".join(step.source_tables)
        reusable_tables = (
            ", ".join(step.reusable_tables)
            if step.reusable_tables
            else "None"
        )

        block = f"""
        Step {step.step_id}
        Target Table: {step.target_table}
        Source Tables: {source_tables}
        Reusable Tables: {reusable_tables}
        Logic: {step.logic}
        """.strip()

        step_blocks.append(block)
        
    formatted_plan = "\n\n".join(step_blocks)
    idiom_examples = "\n".join(
            [f"- {k}: {v}" for k, v in idiom_rules.items()]
        )
 
    # -----------------------------------------
    # Build final system prompt
    # -----------------------------------------
    prompt = system_prompt_sql_executor_template.format(
        idiom=idiom,
        idiom_examples=idiom_examples,
        plan=formatted_plan
    )

    if user_query:
        prompt = prompt + f"\n\n**USER QUERY**:\n{user_query}\n"
        
    return prompt

def planner_prompt_builder( tools:SmartDataTools )->str:
    #txt = data.catalog_snapshot()
    #txt = json.dumps( data.catalog_snapshot(), indent=3)
    txt = tools.catalog_snapshot()
    prompt = system_prompt_sql_planner_template.format(catalog=txt)

    return prompt 



class DataAnalystState(BaseModel):
    user_query: str
    refined_query: Optional[str] = None  
    plan: Optional[ExecutionPlan] = Field(
        default=None,
        description="Structured execution plan generated by planner",
    )

    execution_result: Optional[AgentTableResponse]  = Field(
        default=None,
        description="Executor output",
    )

    error: Optional[str] = Field(
        default=None,
        description="Execution or planning error",
    )

class DataAnalyst:

    def __init__(self, llm, tables_dict: None | Dict[str,pd.DataFrame] = None, 
                            known_table_models: None | Dict[str,TableCard] = None ):

        self._data  : SmartData #= SmartData()
        self._tools : List[StructuredTool] #= SmartDataTools( self._data ).get_tools()
        self._llm = llm 
        self._idiom = 'duckdb'
        self._graph = None 

    
        self._idiom_rules= {'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
        'date truncation': "Use DATE_TRUNC('month', column).",
        'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
        'string concatenation': 'Use the || operator or CONCAT().',
        'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
        'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
        'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)'
        }

        if tables_dict and known_table_models:
            self.initialize_from_known_tables( tables_dict, known_table_models)


    def get_result_as_dataframes(
        self,
        state: DataAnalystState | dict,
    ) -> Dict[str, pd.DataFrame]:
        """
        Retrieve all materialized result tables as DataFrames.

        Supports:
        - DataAnalystState
        - raw graph dict output
        """
        # -----------------------------------------------------
        # Normalize state
        # -----------------------------------------------------
        if isinstance(state, dict):
            state = DataAnalystState(**state)

        results = {}

        if state.execution_result is None:
            return results

        if not state.execution_result.tables:
            return results

        # -----------------------------------------------------
        # Load tables
        # -----------------------------------------------------
        for table_item in state.execution_result.tables:
            table_name = table_item.table_name

            try:
                results[table_name] = self._data.get_table_as_df(
                    table_name
                )

            except Exception as e:
                print(
                    f"Failed loading table '{table_name}': {str(e)}"
                )

        return results

    def initialize_from_known_tables( self, 
                                     tables_dict: Dict[str,pd.DataFrame], 
                                     known_table_models: Dict[str,TableCard] ):
   
        self._data = SmartData() 
        self._data.initialize_from_named_dataframes( tables_dict, known_table_models)
        self._smart_data_tools = SmartDataTools( self._data )#.get_tools()
        self._tools = self._smart_data_tools.get_tools() 
        
    def planner_node( self, state:DataAnalystState):
        
        # generate structured plan
        messages = [] 
        instruction = state.user_query
        planner_prompt = planner_prompt_builder(self._smart_data_tools)
        print(planner_prompt)
        llm = self._llm
        try: 
            messages = [{
                "role": "system",
                "content": planner_prompt
                },
                {
                "role": "user",
                "content": instruction
                }]

            # Structured planner
            structured_llm = llm.with_structured_output(ExecutionPlan)
            plan = structured_llm.invoke(messages)
            #print( plan )


            return state.model_copy(
                update={
                    "plan": plan,
                    #"error": None,
                })


            #return {
            #    #"catalog": catalog,
            #    "plan": plan#.dict() if hasattr(plan, "dict") else plan,
            #}
        
        except Exception as e:
            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })

  
    def execution_node( self, state:DataAnalystState):
  
        llm = self._llm
        plan = state.plan
        idiom= self._idiom
        idiom_rules = self._idiom_rules

        q = plan.user_query if plan.refined_query is None else  plan.refined_query

        executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan,q )

        print("\n" + "=" * 80)
        print("EXECUTOR PROMPT")
        print("=" * 80)
        print(executor_prompt)
        print("=" * 80 + "\n")
        

        agent = create_agent(
                model=llm,
                system_prompt=executor_prompt,
                tools=self._tools,
                response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )
        
        try:
            response = run_agent_stream_values(agent, {} ) 
        

            #print('******************************')
            #print(response)
            #print('******************************')
            


            #return {
            ##"catalog": catalog,
            #"execution_result": response#.dict() if hasattr(plan, "dict") else plan,
            #} 
        
            return state.model_copy(
            update={
                "execution_result": response['structured_response'],
                "error": None,
            })

        
        except Exception as e:
            print("STREAM FAILED:", str(e))


            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })



            #return {
            #"catalog": catalog,
            #"error": str(e)#.dict() if hasattr(plan, "dict") else plan,
            #} 
            
        
    # Conditional after planner
    def should_continue_node(self, state: DataAnalystState):
        if state.error:
            return END

        if state.plan is None:
            return END

        if hasattr(state.plan, "steps") and not state.plan.steps:
            return END

        return "execution"

    # ------------------------------------------------------------------
    # Graph builder
    # ------------------------------------------------------------------
    def as_graph(self):
        builder = StateGraph(DataAnalystState)

        builder.add_node("planner", self.planner_node)
        builder.add_node("execution", self.execution_node)

        builder.set_entry_point("planner")

        builder.add_conditional_edges(
            "planner",
            self.should_continue_node,
            {
                "execution": "execution",
                END: END,
            },
        )

        builder.add_edge("execution", END)

        self._graph = builder.compile()

        return self._graph

    # ------------------------------------------------------------------
    # Main runner
    # ------------------------------------------------------------------
    def run(self, user_query: str):
        """
        Execute full planner -> executor workflow.

        Parameters
        ----------
        user_query : str
            Natural language analytical request.

        Returns
        -------
        DataAnalystState
            Final validated workflow state.
        """
        if self._graph is None:
            self.as_graph()

        initial_state = DataAnalystState(
            user_query=user_query,
            refined_query=None,
            plan=None,
            execution_result=None,
            error=None,
        )

        result = self._graph.invoke(initial_state)
        return self.normalize_state(result)

        # LangGraph may return dict depending on version
        #if isinstance(result, dict):
        #    return DataAnalystState(**result)
        #return result

# ------------------------------------------------------------------
# Normalize output
# ------------------------------------------------------------------
    def normalize_state(self, result) -> DataAnalystState:
        """
        Convert graph output into validated DataAnalystState.
        """
        if isinstance(result, DataAnalystState):
            return result

        if isinstance(result, dict):
            return DataAnalystState(**result)

        raise TypeError(
            f"Unsupported graph output type: {type(result)}"
        )


    # ------------------------------------------------------------------
    # Direct graph invoke wrapper
    # ------------------------------------------------------------------
    def invoke_graph(self, user_query: str) -> DataAnalystState:
        """
        Direct graph call but always returns structured state.
        """
        if self._graph is None:
            self.as_graph()

        result = self._graph.invoke(
            DataAnalystState(
                user_query=user_query,
                refined_query=None,
                plan=None,
                execution_result=None,
                error=None,
            )
        )

        return self.normalize_state(result)


    # ------------------------------------------------------------------
    # Export as LangChain tool
    # ------------------------------------------------------------------
    def as_tool(self) -> StructuredTool:
        """
        Expose DataAnalyst as a reusable tool for other agents.
        """

        def _run_analysis(user_query: str) -> dict:
            state = self.run(user_query)
            return state 
        
            return {
                "plan": (
                    state.plan.model_dump()
                    if state.plan else None
                ),
                "execution_result": (
                    state.execution_result.model_dump()
                    if state.execution_result else None
                ),
                "error": state.error,
            }

        return StructuredTool.from_function(
            func=_run_analysis,
            name="data_analyst",
            description=(
                "Executes structured analytical workflows over known tabular datasets. "
                "Useful for SQL-style table generation, ranking, aggregations, "
                "time-series analysis, and derived table creation."
            ),
        )




In [ ]:
analyst = DataAnalyst( llm, df_dict, known_table_models )

tool = analyst.as_tool()
print(tool)


# =========================================================
# Direct tool invocation
# =========================================================
tool_result = tool.invoke(
    {
        "user_query": (
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    }
)
print("TOOL RESULT:")
type(tool_result)
tool_result.execution_result.tables





graph = analyst.as_graph() 
display( graph )

# =========================================================
# Direct graph invoke
# =========================================================
result = graph.invoke(
    DataAnalystState(
        user_query=(
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    )
)

# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)




# =========================================================
# Simple invoke test
# =========================================================
response = analyst.run(
    "Create two separate tables: "
    "one ranking injector wells by total water injection volume, "
    "and another ranking producer wells by total oil production volume."
)

print("FINAL RESPONSE:")
#print(response)



In [ ]:

graph = analyst.as_graph() 
display( graph )

# Direct graph invoke
# =========================================================
result = graph.invoke(
    DataAnalystState(
        user_query=(
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    )
)

# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]

display(first_df)

In [ ]:
# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]

display(first_df)

In [ ]:
type(result)
result.keys()
type(result['execution_result'])
state = DataAnalystState(**result)
state

In [ ]:

# =========================================================
# Simple invoke test
# =========================================================
response = analyst.run(
    "Create two separate tables: "
    "one ranking injector wells by total water injection volume, "
    "and another ranking producer wells by total oil production volume."
)

print("FINAL RESPONSE:")
#print(response)



In [ ]:
type(response)

In [ ]:
# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(response)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]

display(first_df)


In [ ]:
if response.plan:
    print("\nPLAN:")
    print(response.plan.model_dump_json(indent=3))


if response.execution_result:#" in response:
    print("\nEXECUTION RESULT:")
    print(response.execution_result.model_dump_json(indent=3))

In [ ]:
response.execution_result.tables

In [ ]:

executor_prompt = executor_prompt_builder(idiom, idiom_rules,plan)

agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        #response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))


In [ ]:
tools

In [ ]:




class old:     

    def fdgdfgrun( self, user_query):

        llm = self._llm
        #planner
        planner_prompt = self._planner_prompt_builder()
        messages = [{
            "role": "system",
            "content": planner_prompt
            },
            {
            "role": "user",
            "content": user_query
            }]
        structured_llm = llm.with_structured_output(ExecutionPlan)

        plan = structured_llm.invoke(messages)
        print('Plan result')
        print( plan )
        
        executor_prompt = self._executor_prompt_builder(self._idiom, 
                                                        self._idiom_rules, 
                                                        plan )  
        executor_agent = create_agent(
                model=llm,
                system_prompt=executor_prompt,
                tools=self._tools,
                #response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )   
         
        

        #executor 
        try:
            response = run_agent_stream_values(executor_agent, {} ) 
            last_response = response 
        except Exception as e:
            print("STREAM FAILED:", str(e))





    def xxrun(self, user_query: str) -> Dict[str, Any]:

        initial_state: DataAnalystState = {
            "user_query": user_query,
            "plan": None,
            "execution_result": None,
            "error": None,
        }

        return self._graph.invoke(initial_state)    
    
    def planner_node(self,state: DataAnalystState) -> DataAnalystState:
        
        llm = self._llm
        
        # Structured planner
        structured_planner = llm.with_structured_output(ExecutionPlan)
        state["error"] = "" 
        try:
            planner_prompt = self._planner_prompt_builder()
            instruction = state['user_query']
            messages = [
                {
                    "role": "system",
                    "content": planner_prompt
                },
                {
                    "role": "user",
                    "content": instruction
                }
            ]

                
            
            plan = structured_planner.invoke(messages)
            
            state["plan"] = plan
            print("Plan produced")

        except Exception as e:
            state["error"] = f"Planner failed: {str(e)}"

        return state

    def executor_node(self,state: DataAnalystState) -> DataAnalystState:

        llm = self._llm
        if state.get("error"):
            return state

        try:
            plan = state["plan"]
            executor_system_prompt = self._executor_prompt_builder(self._idiom, 
                                                                   self._idiom_rules, 
                                                                   plan)
            
            executor_agent = create_agent(
                model=llm,
                system_prompt=executor_system_prompt,
                tools=self._tools,
                #response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )     
            
            
            result = executor_agent.invoke({
                "messages": [
                    {
                        "role": "system",
                        "content": executor_system_prompt
                    }

                ]
            })

            state["execution_result"] = result

        except Exception as e:
            state["execution_result"] = None 
            state["error"] = f"Executor failed: {str(e)}"

        return state







        return state 

    def _build_graph(self):

        builder = StateGraph(DataAnalystState)

        builder.add_node("planner", self.planner_node)
        builder.add_node("executor", self.executor_node)

        builder.set_entry_point("planner")

        builder.add_conditional_edges(
            "planner",
            self._should_continue,
            {
                "executor": "executor",
                "end": END,
            }
        )

        builder.add_edge("executor", END)

        return builder.compile()

    def _should_continue(
        self,
        state: DataAnalystState
    ) -> str:

        if state.get("error"):
            return "end"

        return "executor"
    
    def _planner_prompt_builder( self )->str:
        txt = self._data.catalog_snapshot()
        txt = json.dumps( self._data.catalog_snapshot(), indent=3)

        prompt = system_prompt_sql_planner_template.format(catalog=txt)

        return prompt 

    def _executor_prompt_builder( self, idiom, idiom_examples, plan )->str:
        prompt = system_prompt_sql_executor_template.format(idiom=idiom, idiom_examples=idiom_examples, plan=plan)
        return prompt 



        





In [ ]:
analyst.run( query9 )


In [ ]:
display(analyst._graph )


In [ ]:
for n,item in enumerate(queries):

    print("\n\n")
    print(120*'=')
    user_query = item[0]
    print( user_query, 30*' ', n  )
    print(120*'=')
    

    #user_query = "name the first tree wells in the last table"
    messages = {"messages": [{"role": "user", "content": user_query}]}
    
    try:
        response = run_agent_stream_values(agent, messages ) 
        last_response = response 
    except Exception as e:
        print("STREAM FAILED:", str(e))
    #response = agent.invoke(
    #messages,
    #config={"recursion_limit": 10},
    #)
    print(50*'=',sep="\n\n")
    break